# Plantilla Base MLOps (MLflow + lakeFS)

Este notebook es una plantilla reutilizable para todos los equipos.

Flujo:
1. Configurar identificadores del caso de uso.
2. Subir dataset a lakeFS y obtener commit hash.
3. Entrenar y registrar ejecuciones en MLflow con trazabilidad.
4. Comparar configuraciones de hiperparametros.

Uso:
- Antes de completar los TODOs, revisar el documento convencion.md:
- Completar los TODOs marcados en el código.
- Ejecutar y verificar en JupyterHub y en la UI de MLflow.

## 1. Imports

In [ ]:
import os
from datetime import date

import lakefs_sdk
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from lakefs_sdk.client import LakeFSClient

# TODO [1/6]: Importar las librerias necesarias del modelo
# Ejemplo: from sklearn.ensemble import RandomForestRegressor

# TODO [2/6]: Importar metricas y split segun el tipo de modelo
# Ejemplo:
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from sklearn.model_selection import train_test_split

## 2. Configuracion global

Ajustar solo los campos TODO.

Importante:
- Las variables de entorno ya están inyectadas en JupyterHub por lo que no es necesario modificarlas.
- Usar los nombres definidos convencion.md.

In [ ]:
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LAKEFS_HOST = os.environ.get("LAKEFS_ENDPOINT", "http://lakefs:8000")
LAKEFS_ACCESS = os.environ.get("LAKEFS_ACCESS_KEY_ID")
LAKEFS_SECRET = os.environ.get("LAKEFS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(MLFLOW_URI)

# TODO [3/6]: Configurar los identificadores del caso de uso
# Consulta MLOps/compose/convencion.md para los valores correctos.
#
# EXPERIMENT_NAME: nombre del experimento en MLflow
# CASO_USO: letra del caso (B, C, D, E...)
# GRUPO: grupo (G1, G3, G4...)
# DATASET_REPO: repositorio en lakeFS
# REGISTERED_MODEL: nombre oficial en Model Registry
# VARIABLE_TARGET: columna objetivo a predecir
#
# Repositorios por caso (referencia):
#   Caso B: uci-appliances
#   Caso C: lbnl-fdd
#   Caso D: uci-occupancy
#   Caso E: era5
EXPERIMENT_NAME = "CasoX_DATASET_ALGO" # TODO: cambiar por el del caso de uso
CASO_USO = "X" # TODO: cambiar por la letra del caso de uso
GRUPO = "G1" # TODO: cambiar por el grupo
DATASET_REPO = "nombre-repo-lakefs" # TODO: cambiar por el repositorio
REGISTERED_MODEL = "simarro-caso-x-modelo" # TODO: cambiar por el modelo
VARIABLE_TARGET = "target" # TODO: cambiar por la variable objetivo del modelo

## 3. Funciones del pipeline

Incluye subida a lakeFS, carga de datos y entrenamiento con registro en MLflow.

In [ ]:
def subir_dataset(ruta_csv: str, rama: str = "dev") -> str:
    """
    Sube el CSV del dataset a lakeFS en la rama indicada y hace commit.
    Devuelve el commit hash para usarlo como dataset_version en MLflow.

    Args:
        ruta_csv: ruta local al fichero CSV (ej: '/home/casoD/datos.csv')
        rama: rama de lakeFS donde subir (por defecto 'dev')
    """
    cfg = lakefs_sdk.Configuration(
        host=LAKEFS_HOST,
        username=LAKEFS_ACCESS,
        password=LAKEFS_SECRET,
    )
    client = LakeFSClient(configuration=cfg)

    # TODO [4/6]: Cambia el nombre del fichero destino en lakeFS.
    # Formato recomendado: data/[nombre_descriptivo].csv
    # Ejemplos por caso:
    #   Caso B: data/uci_appliances.csv
    #   Caso C: data/lbnl_fdd.csv
    #   Caso D: data/uci_occupancy.csv
    #   Caso E: data/era5_xativa.csv
    NOMBRE_FICHERO_LAKEFS = "data/dataset.csv"

    with open(ruta_csv, "rb") as f:
        client.objects_api.upload_object(
            repository=DATASET_REPO,
            branch=rama,
            path=NOMBRE_FICHERO_LAKEFS,
            content=f,
        )

    commit = client.commits_api.commit(
        repository=DATASET_REPO,
        branch=rama,
        commit_creation=lakefs_sdk.CommitCreation(
            message=f"feat: dataset {DATASET_REPO} subido por {GRUPO}",
            metadata={"equipo": GRUPO, "caso_uso": CASO_USO},
        ),
    )

    print("Dataset subido a lakeFS")
    print(f"  Repositorio: {DATASET_REPO}")
    print(f"  Rama: {rama}")
    print(f"  Commit hash: {commit.id}")
    return commit.id


def load_data(ruta_csv: str) -> tuple:
    """
    Cargar el dataset y dividirlo en train/test.

    TODO [5/6]: Adaptar esta funcion al dataset del caso de uso.

    Debes:
      a) Leer CSV con pandas
      b) Separar features (X) y variable objetivo (y)
      c) Eliminar columnas innecesarias (timestamps, IDs, etc.)
      d) Tratar nulos, outliers, etc (fillna, dropna, etc.)

    Return: (X_train, X_test, y_train, y_test)
    """
    df = pd.read_csv(ruta_csv)

    # Ejemplo Caso D (uci_occupancy):
    # y = df['Occupancy']
    # X = df.drop(columns=['Occupancy', 'date'])

    y = df[VARIABLE_TARGET]
    X = df.drop(columns=[VARIABLE_TARGET])
    X = X.select_dtypes(include=[np.number])
    X = X.fillna(X.median())

    return train_test_split(X, y, test_size=0.2, random_state=42)


def registrar_report_evidently(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    caso: str,
    dataset: str,
    commit: str,
) -> None:
    """
    Genera un informe básico de drift con Evidently y lo adjunta como artefacto en MLflow
    """
    try:
        from evidently import Report
        from evidently.presets import DataDriftPreset, DataSummaryPreset
        import tempfile
    except Exception as exc:
        print(f"WARN: Evidently no se encuentra disponible: {exc}")
        mlflow.set_tag("evidently_status", "unavailable")
        return

    try:
        current = X_test.copy()
        reference = X_train.copy()

        # Evita errores y problemas con columnas no serializables
        for col in reference.columns:
            if str(reference[col].dtype).startswith("datetime"):
                reference[col] = reference[col].astype(str)
                if col in current.columns:
                    current[col] = current[col].astype(str)

        report = Report(metrics=[DataSummaryPreset(), DataDriftPreset()])
        snapshot = report.run(reference_data=reference, current_data=current)

        with tempfile.TemporaryDirectory(prefix="evidently_") as tmp_dir:
            html_path = os.path.join(tmp_dir, "evidently_report.html")
            json_path = os.path.join(tmp_dir, "evidently_report.json")

            snapshot.save_html(html_path)
            snapshot.save_json(json_path)

            mlflow.log_artifact(html_path, artifact_path="monitoring/evidently")
            mlflow.log_artifact(json_path, artifact_path="monitoring/evidently")

        mlflow.set_tags(
            {
                "evidently_status": "ok",
                "evidently_reference_rows": str(len(reference)),
                "evidently_current_rows": str(len(current)),
                "evidently_case": caso,
                "evidently_dataset": dataset,
                "evidently_dataset_version": commit,
            }
        )
        print("Reporte Evidently registrado en MLflow")

    except Exception as exc:
        print(f"WARN: No se pudo generar report Evidently: {exc}")
        mlflow.set_tag("evidently_status", "error")
        mlflow.set_tag("evidently_error", str(exc)[:250])


def train_registry(ruta_csv: str, commit_hash: str, params: dict) -> str:
    """
    Entrena el modelo y registra el run completo en MLflow con trazabilidad.

    Args:
        ruta_csv: ruta local al CSV
        commit_hash: commit de lakeFS para dataset_version
        params: hiperparámetros del modelo
    """
    mlflow.set_experiment(EXPERIMENT_NAME)
    run_name = f"RF_{date.today().strftime('%Y%m%d')}_baseline"

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"Run iniciado: {run_id}")
        print(f"Experimento: {EXPERIMENT_NAME}")
        print(f"Nombre run: {run_name}\n")

        # Tags de trazabilidad obligatorios (ver MLOps/compose/convencion.md)
        mlflow.set_tags(
            {
                "caso_uso": CASO_USO,
                "grupo": GRUPO,
                "dataset": DATASET_REPO,
                "dataset_version": commit_hash,
                "dataset_branch": "dev",
                "capa_medallion": "silver", # TODO: ajustar segun la capa del dataset
                "ejecutado_por": os.environ.get("JUPYTERHUB_USER", "local"),
            }
        )

        # Registrar hiperparámetros
        mlflow.log_params(params)

        # Cargar datos y registrar metadatos del dataset
        X_train, X_test, y_train, y_test = load_data(ruta_csv)
        mlflow.set_tags(
            {
                "n_train": len(X_train),
                "n_test": len(X_test),
                "features": ", ".join(X_train.columns.tolist()),
            }
        )

        # Entrenamiento
        # TODO [6/6]: Cambiar modelo y parámetros según el caso de uso
        # Ejemplos:
        #   Regresion: RandomForestRegressor(**params)
        modelo = RandomForestRegressor(**params)  # TODO: cambiar por el modelo
        modelo.fit(X_train, y_train)

        # Evaluación
        predicciones = modelo.predict(X_test)

        # Ajustar métricas
        # TODO: sustituir por las métricas según el tipo de modelo (ver TODO [2/6])
        rmse = mean_squared_error(y_test, predicciones) ** 0.5
        mae = mean_absolute_error(y_test, predicciones)
        r2 = r2_score(y_test, predicciones)

        mlflow.log_metrics(
            {
                "rmse": round(rmse, 4),
                "mae": round(mae, 4),
                "r2": round(r2, 4),
            }
        )

        # Registro del modelo en Model Registry
        model_info = mlflow.sklearn.log_model(
            sk_model=modelo,
            artifact_path="model",
            registered_model_name=REGISTERED_MODEL,
            metadata={
                "caso_uso": CASO_USO,
                "framework": "scikit-learn",
                "task": "regression",  # TODO: cambiar a classification si aplica
            },
        )

        print(f"Modelo registrado: {REGISTERED_MODEL}")
        print(f"URI: {model_info.model_uri}")

        # Guardar predicciones como artefacto para auditoría
        df_pred = pd.DataFrame(
            {
                "real": y_test.values,
                "prediccion": predicciones,
                "error": y_test.values - predicciones,
            }
        )
        df_pred.to_csv("/tmp/predicciones.csv", index=False)
        mlflow.log_artifact("/tmp/predicciones.csv", artifact_path="evaluacion")

        return run_id

## 4. Ejecucion del experimento

Actualizar la ruta del CSV y configurar los hiperparámetros del modelo.

Cuando los resultados sean satisfactorios, hacer merge dev --> main para disparar el pipeline automático en la UI de lakeFS.

In [ ]:
def ejecutar_experimento():
    # TODO: ajustar esta ruta a la del fichero en el entorno JupyterHub
    RUTA_CSV = "/home/casoX/dataset.csv"

    # Paso 1: subir dataset a lakeFS y guardar commit para trazabilidad
    print("=" * 75)
    print("PASO 1: Subiendo dataset a lakeFS...")
    print("=" * 75)
    commit_hash = subir_dataset(RUTA_CSV, rama="dev")

    # TODO: ajusta estas configuraciones al modelo elegido
    configuraciones = [
        {"n_estimators": 50, "max_depth": 3, "random_state": 42},
        {"n_estimators": 100, "max_depth": 5, "random_state": 42},
        {"n_estimators": 200, "max_depth": 10, "random_state": 42},
    ]

    # Paso 2: entrenar y registrar cada configuracion en MLflow
    print("\n" + "=" * 75)
    print("PASO 2: Entrenando y registrando en MLflow...")
    print("=" * 75)

    run_ids = []
    for i, params in enumerate(configuraciones, 1):
        print(f"\n[{i}/{len(configuraciones)}] Configuracion: {params}")
        run_id = train_registry(RUTA_CSV, commit_hash, params)
        run_ids.append(run_id)

    print("\n" + "=" * 75)
    print("Experimento completado.")
    print(f"  Revisar en MLflow: {MLFLOW_URI}")
    print(f"  Experimento: {EXPERIMENT_NAME}")
    print("\nSIGUIENTE PASO:")
    print("  Si los resultados son correctos, hacer merge de dev a main en lakeFS")
    print("  para disparar el pipeline de reentrenamiento automático.")
    print("=" * 75)

    return run_ids

In [ ]:
# TODO: Descomentar las siguientes líneas cuando estén completados los TODOs
# run_ids = ejecutar_experimento()
# run_ids